In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
from tqdm import tqdm
import datetime as dt

C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
dtm_now = dt.datetime.today()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 14:38:45.066326


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'
# dict
list_str_replace = [
    '15_in_60',
    '30_in_90',
    '30_in_180',
    '30_in_360',
    '60_in_720',
]

Project: 20250307-funded-trends
Task: 03_pull_targets
Subtask: 01_classification


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### DB Connection

In [6]:
# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

### Get DPD targets

In [7]:
# read query
str_filepath = './sql/query_dpd.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT \n'
 '\tam.bigAccountId,\n'
 '\tCASE WHEN Early_Pay_Delinquency_STRDPD_STRTOTAL > 0 THEN 1 ELSE 0 END AS '
 'Early_Pay_Delinquency_STRDPD_STRTOTAL_Flag\n'
 'FROM \n'
 '\telectra.pfsdb.dbo.tblAccountMaintenance am LEFT OUTER JOIN\n'
 '\t(\n'
 '\t\tSELECT \n'
 '\t\t\tc.bigAccountId,\n'
 '\t\t\tSUM(CASE\n'
 '\t\t\t\t\tWHEN DATEDIFF(DAY, c.dtmDue, c.dtmClosed) > DPDMINUS1 OR '
 '(DATEDIFF(DAY, c.dtmDue, GETDATE()) > DPDMINUS1 AND c.dtmClosed IS NULL) \n'
 '\t\t\t\t\tTHEN 1\n'
 '\t\t\t\t\tELSE 0\n'
 '\t\t\t\tEND) AS Early_Pay_Delinquency_STRDPD_STRTOTAL\n'
 '\t\tFROM pfsdb.dbo.tblCharges c LEFT OUTER JOIN pfsdb.dbo.tblAccountTerms '
 'att ON c.bigAccountTermId = att.bigAccountTermId\n'
 '\t\t\tINNER JOIN pfsdb.dbo.tblAccountMaintenance am ON c.bigAccountId = '
 'am.bigAccountId\n'
 '\t\tWHERE c.bigChargeTypeId = 1 AND c.bitInvalid = 0 AND c.dtmDue < '
 'DATEADD(DAY, STRTOTAL, am.dtmContract)\n'
 '\t\tGROUP BY c.bigAccountId\n'
 '\t)a\n'
 '\tON am.bigAccountId = a.bigAccountId')


In [8]:
list_df = []
for str_replace in tqdm(list_str_replace):
    # DPD
    STRDPD = str_replace.split('_in_')[0]
    # total
    STRTOTAL = str_replace.split('_in_')[1]
    # DPD minus 1
    STRDPDMINUS1 = str(int(STRDPD) - 1)
    
    # replace
    str_query_tmp = str_query.replace('STRDPD', STRDPD)
    str_query_tmp = str_query_tmp.replace('STRTOTAL', STRTOTAL)
    str_query_tmp = str_query_tmp.replace('DPDMINUS1', STRDPDMINUS1)
    
    # pull from db
    df = pd.read_sql_query(
        str_query_tmp, 
        con=conn,
    )
    # set index
    df = df.set_index('bigAccountId')
    # append
    list_df.append(df)

# join
df = pd.concat(list_df, axis=1, join='outer')
# reset index
df.reset_index(inplace=True)
# show
df

  0%|                                                                                            | 0/5 [00:00<?, ?it/s]C:\Users\aengland\AppData\Local\Temp/ipykernel_20776/2043433972.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(
100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:16<00:00, 15.40s/it]


,bigAccountId,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag
0,6,0,0,0,0,0
1,25,1,0,0,1,1
2,73,0,0,0,0,1
3,82,1,1,1,1,1
4,122,0,0,0,1,1
...,...,...,...,...,...,...
344643,8713064,0,0,0,0,0
344644,8713405,0,0,0,0,0
344645,8715756,0,0,0,0,0
344646,8716150,0,0,0,0,0


### Close connection

In [9]:
# close
conn.close()

### Make run date column to get days on books

In [10]:
df['run_date'] = dtm_now
# show
df

,bigAccountId,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date
0,6,0,0,0,0,0,2025-03-10 14:38:45.066326
1,25,1,0,0,1,1,2025-03-10 14:38:45.066326
2,73,0,0,0,0,1,2025-03-10 14:38:45.066326
3,82,1,1,1,1,1,2025-03-10 14:38:45.066326
4,122,0,0,0,1,1,2025-03-10 14:38:45.066326
...,...,...,...,...,...,...,...
344643,8713064,0,0,0,0,0,2025-03-10 14:38:45.066326
344644,8713405,0,0,0,0,0,2025-03-10 14:38:45.066326
344645,8715756,0,0,0,0,0,2025-03-10 14:38:45.066326
344646,8716150,0,0,0,0,0,2025-03-10 14:38:45.066326


### Save

In [11]:
%%time

# save
str_filename = 'df_targets.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 747 ms


### Upload to s3

In [12]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 372 ms


### Clean-up

In [13]:
os.remove(str_local_path)